# KLE Colab Notebook — Walk-Forward Ensemble

核心改进：使用 **Walk-Forward Ensemble** 策略提升预测命中率。

## 策略说明
| 旧方式 | 新方式 |
|--------|--------|
| 一次性训练全部历史数据 | 滚动训练：每次预测只用截止当期的历史 |
| 单一模型单一 seq_len | 多 seq_len × 多 seed 集成，取平均概率 |
| 无真实泛化评估 | 先 walk-forward 回测验证，再训练最终预测 |

## 运行流程
1. 安装依赖 + 克隆仓库
2. 检查数据
3. **Walk-forward 回测**：在最近 N 期上模拟真实预测，评估命中率
4. 可视化回测结果
5. **最终预测**：用全量数据训练 ensemble，输出下一期号码

建议在 Colab 中把运行时切换到 `GPU`。

In [ ]:
import os
import sys
import subprocess
from pathlib import Path


def run(cmd: str) -> None:
    print(f"+ {cmd}")
    subprocess.run(cmd, shell=True, check=True)


REPO_URL = "https://github.com/jursmatsko/lotto-image-predictor-research.git"
REPO_ROOT = Path("/content/lotto-image-predictor-research")
PROJECT_DIR = REPO_ROOT / "kle"

run("python -m pip install -q pandas matplotlib requests beautifulsoup4 lxml")

try:
    import torch
except ImportError:
    run("python -m pip install -q torch")
    import torch

if not REPO_ROOT.exists():
    run(f"git clone {REPO_URL} {REPO_ROOT}")
else:
    print(f"Repo already exists: {REPO_ROOT}")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f"Python: {sys.version.split()[0]}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 数据检查

先确认 `data/data.csv` 能正确读取，并检查最近几期数据结构。

In [ ]:
import pandas as pd


data_path = PROJECT_DIR / "data" / "data.csv"
df = pd.read_csv(data_path)

print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
display(df.head())
display(df.tail())

## 加载应用

这里直接复用仓库里的 `ImagePredictionApp`，但训练模型选择 `pytorch_cnn`，因为它在 Colab 的兼容性和速度都更好。

In [ ]:
from image_predictor.main import ImagePredictionApp
from image_predictor.models.pytorch_image_cnn import DEVICE


app = ImagePredictionApp()
app.loader.load_data()

print(f"Training device: {DEVICE}")
print(app.loader.get_statistics())

## 参数配置

| 参数 | 作用 |
|------|------|
| `SEQ_LENS` | 每个集成成员使用的历史序列长度，越多越丰富但越慢 |
| `MODELS_PER_SEQ` | 每个 seq_len 训练几个不同 seed 的模型 |
| `WF_EPOCHS` | Walk-forward 回测中每个模型的训练轮数（建议 20-30） |
| `FINAL_EPOCHS` | 最终预测时每个模型的训练轮数（建议 40-60） |
| `WF_EVAL_LAST` | 在最近多少期上做回测评估 |

**总模型数 = len(SEQ_LENS) × MODELS_PER_SEQ**，越多越准但越慢。

In [ ]:
SEQ_LENS        = [10, 15, 20]  # 每个集成成员的历史窗口长度
MODELS_PER_SEQ  = 2             # 每个 seq_len 用几个不同 seed 训练
WF_EPOCHS       = 25            # walk-forward 回测每个模型的训练轮数
FINAL_EPOCHS    = 50            # 最终预测集成每个模型的训练轮数
LEARNING_RATE   = 7e-4          # 学习率
WF_EVAL_LAST    = 20            # 在最近多少期上做回测
NUM_TICKETS     = 10            # 生成几注推荐号码
BASE_SEED       = 20260313      # 随机种子基数

MODEL_CACHE_DIR = PROJECT_DIR / "image_predictor" / "models" / "saved"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

n_ensemble = len(SEQ_LENS) * MODELS_PER_SEQ
print(f"Ensemble size per step: {n_ensemble} models")
print(f"seq_lens={SEQ_LENS}, models_per_seq={MODELS_PER_SEQ}")
print(f"wf_epochs={WF_EPOCHS}, final_epochs={FINAL_EPOCHS}, lr={LEARNING_RATE}")
print(f"walk-forward eval on last {WF_EVAL_LAST} draws")

In [ ]:
import numpy as np
import torch
from image_predictor.models.pytorch_image_cnn import PyTorchImagePredictor


def train_one_model(train_imgs, seq_len, epochs, lr, seed):
    """在给定历史图像上训练一个 PyTorch CNN，返回训练好的 predictor。"""
    np.random.seed(seed)
    torch.manual_seed(seed)

    X, y = [], []
    for i in range(seq_len, len(train_imgs)):
        X.append(train_imgs[i - seq_len:i])
        y.append(train_imgs[i])
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    val_size = max(1, int(len(X) * 0.15))
    X_train, X_val = X[:-val_size], X[-val_size:]
    y_train, y_val = y[:-val_size], y[-val_size:]

    model = PyTorchImagePredictor(seq_len=seq_len, hidden_channels=64)
    model.fit(X_train, y_train, X_val, y_val,
              epochs=epochs, lr=lr, verbose=False)
    return model


# ── 准备数据 ──────────────────────────────────────────────
app.loader.load_data()
images = app.loader.create_images().astype(np.float32)
issues = app.loader.issues
encoder = app.encoder
min_train = max(SEQ_LENS) + 50

eval_start = max(min_train, len(images) - WF_EVAL_LAST)
n_steps = len(images) - eval_start
print(f"Total draws: {len(images)}")
print(f"Walk-forward steps: {n_steps}  (draws {eval_start} → {len(images)-1})")
print(f"Ensemble per step: {len(SEQ_LENS)} seq_lens × {MODELS_PER_SEQ} seeds = {len(SEQ_LENS)*MODELS_PER_SEQ} models")
print("=" * 60)

# ── Walk-forward 回测 ─────────────────────────────────────
wf_results = []
for step, test_idx in enumerate(range(eval_start, len(images))):
    train_imgs = images[:test_idx]
    preds = []

    for seq_len in SEQ_LENS:
        if len(train_imgs) <= seq_len + 30:
            continue
        for m in range(MODELS_PER_SEQ):
            seed = BASE_SEED + test_idx * 10 + seq_len * 100 + m
            model = train_one_model(train_imgs, seq_len, WF_EPOCHS, LEARNING_RATE, seed)
            latest = train_imgs[-seq_len:][np.newaxis, ...]
            preds.append(model.predict(latest)[0])

    ensemble_pred = np.mean(np.stack(preds, axis=0), axis=0)
    pred_nums  = set(encoder.decode_single(ensemble_pred, 20))
    actual_nums = set(encoder.decode_single(images[test_idx], 20))
    hits = len(pred_nums & actual_nums)

    wf_results.append({
        "issue": issues[test_idx],
        "hits": hits,
        "predicted": sorted(pred_nums),
        "actual":    sorted(actual_nums),
    })
    print(f"[{step+1:>3}/{n_steps}] Draw {issues[test_idx]}: {hits:>2}/20 hits  "
          f"({'▲' if hits > 5 else '▼' if hits < 5 else '─'})")

## Walk-Forward 回测结果

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

wf_df = pd.DataFrame(wf_results)
avg   = wf_df["hits"].mean()
above = (wf_df["hits"] > 5).sum()

print("=" * 50)
print(f"  Average hits  : {avg:.2f} / 20")
print(f"  Random baseline: 5.00 / 20")
print(f"  vs Random      : {avg - 5:+.2f}")
print(f"  Draws > 5 hits : {above} / {len(wf_df)}")
print("=" * 50)
display(wf_df[["issue", "hits"]])

# ── 柱状图 ───────────────────────────────────────────────
colors = ["#2ecc71" if h > 5 else "#e74c3c" if h < 5 else "#f39c12"
          for h in wf_df["hits"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(len(wf_df)), wf_df["hits"], color=colors)
axes[0].axhline(5.0, color="orange", linestyle="--", linewidth=1.5, label="Random (5.0)")
axes[0].axhline(avg, color="royalblue", linestyle="-", linewidth=2, label=f"Average ({avg:.2f})")
axes[0].set_xlabel("Draw index")
axes[0].set_ylabel("Hits / 20")
axes[0].set_title("Walk-Forward Ensemble: Hits per Draw")
axes[0].legend()

green_p = mpatches.Patch(color="#2ecc71", label="> 5 hits")
red_p   = mpatches.Patch(color="#e74c3c", label="< 5 hits")
axes[0].legend(handles=[green_p, red_p,
    plt.Line2D([], [], color="orange", linestyle="--", label="Random"),
    plt.Line2D([], [], color="royalblue", label=f"Avg {avg:.2f}")
])

# 命中分布直方图
axes[1].hist(wf_df["hits"], bins=range(0, 21), color="#3498db", edgecolor="white", align="left")
axes[1].axvline(5.0, color="orange", linestyle="--", linewidth=1.5, label="Random (5.0)")
axes[1].axvline(avg,  color="royalblue", linestyle="-",  linewidth=2,   label=f"Avg ({avg:.2f})")
axes[1].set_xlabel("Hits")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Hit Distribution")
axes[1].legend()

plt.tight_layout()
plt.savefig(str(MODEL_CACHE_DIR / "wf_results.png"), dpi=120, bbox_inches="tight")
plt.show()

## 最终预测 — 用全量数据训练 Ensemble

回测只用来评估策略质量。最终预测用**全量历史数据**重新训练一批 ensemble 模型，集成后给出下一期推荐。

In [ ]:
print("Training final ensemble on full dataset …")
print(f"  {len(SEQ_LENS)} seq_lens × {MODELS_PER_SEQ} models = {len(SEQ_LENS)*MODELS_PER_SEQ} total")

final_preds = []
for seq_len in SEQ_LENS:
    for m in range(MODELS_PER_SEQ):
        seed = BASE_SEED + 88888 + seq_len * 100 + m
        model = train_one_model(images, seq_len, FINAL_EPOCHS, LEARNING_RATE, seed)
        latest = images[-seq_len:][np.newaxis, ...]
        final_preds.append(model.predict(latest)[0])
        print(f"  ✓ seq_len={seq_len}  seed={m+1}")

ensemble_final = np.mean(np.stack(final_preds, axis=0), axis=0)
top20 = encoder.decode_single(ensemble_final, 20)

print(f"\n{'='*60}")
print(f"  Ensemble size : {len(final_preds)} models")
print(f"  Top 20        : {top20}")
print(f"{'='*60}")

# 概率热力图
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(ensemble_final, cmap="hot", vmin=0, vmax=ensemble_final.max(), aspect="equal")
fig.colorbar(im, ax=ax, label="Probability")
ax.set_xticks(range(10));  ax.set_xticklabels(range(1, 11))
ax.set_yticks(range(8));   ax.set_yticklabels([f"{r*10+1}-{r*10+10}" for r in range(8)])
for r in range(8):
    for c in range(10):
        num = r * 10 + c + 1
        is_top = num in set(top20)
        val = ensemble_final[r, c]
        ax.text(c, r, str(num),
                ha="center", va="center", fontsize=8, fontweight="bold",
                color="cyan" if is_top else ("white" if val > ensemble_final.mean() else "gray"))
ax.set_title(f"Ensemble Probability Map — Top20 highlighted in cyan")
plt.tight_layout()
heatmap_path = str(MODEL_CACHE_DIR / "ensemble_heatmap.png")
plt.savefig(heatmap_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"Saved: {heatmap_path}")

In [ ]:
tickets = app._generate_tickets(ensemble_final, NUM_TICKETS)

print(f"{'='*50}")
print(f"  Next Draw Prediction")
print(f"  Top 20: {' '.join(f'{n:02d}' for n in top20)}")
print(f"{'='*50}")
print(f"\n  {NUM_TICKETS} Recommended Tickets (10 numbers each):")
for i, ticket in enumerate(tickets, 1):
    print(f"  Ticket {i:02d}: {' '.join(f'{n:02d}' for n in ticket)}")

print(f"\n  Walk-Forward Backtest Summary ({WF_EVAL_LAST} draws):")
print(f"  Average Hits : {avg:.2f}/20  (Random baseline: 5.00/20)")
print(f"  vs Random    : {avg-5:+.2f}")
print(f"\n  ⚠️  Lottery is random. This is for scientific research only.")

## 可选：保存到 Google Drive

如果你想长期保留模型文件，可以在 Colab 中挂载 Drive 后，把 `MODEL_PATH` 改到你的 Drive 目录，例如：

```python
from google.colab import drive
drive.mount('/content/drive')
MODEL_PATH = Path('/content/drive/MyDrive/kle_models/colab_pytorch_cnn.pt')
```

之后重新运行训练单元即可。